## 한 프로세스를 만드는 과정

### 1. `STAGE` 및 `STREAM` 생성
- `Stage`에 `Directory` 활성화(파일 메타데이터를 테이블처럼 조회가 가능하기 때문에)
    - 나중에는 `Directory`라는 메서드를 사용해서 batch가 가능함
- `Stream`은 `Stage`에 `image`를 넣을 때의 이벤트를 감지하는 역할을 함

In [ ]:
-- Stage 생성
CREATE OR REPLACE STAGE {DB}.{SCHEMA}.EVENT_STAGE
DIRECTORY = (ENABLE = TRUE);

-- 위에서 만든 EVENT_STAGE에 image_stage_stream라는 Stream을 생성함
CREATE OR REPLACE STREAM image_stage_stream ON STAGE EVENT_STAGE;

### 2. `Cortex Search`가 참조하는 테이블을 생성
- ocr_batch시 코텍스 결과(`create_cortex_refer.py`에서 만든 결과)를 참조하여 결과값을 만듦

In [ ]:
-- 코텍스 참조 테이블 생성
CREATE OR REPLACE TABLE {DB}.{SCHEMA}.COTEX_SEARCH_REFER_TABLE (
    ID VARCHAR(64),
    RAW_TEXT STRING, -- 입력
    ITEM_CATEGORY STRING, -- 품목
    FINAL_JSON STRING  -- 출력
);

-- 참조 방식 설정
CREATE OR REPLACE CORTEX SEARCH SERVICE {DB}.{SCHEMA}.COTEX_SEARCH_REFER_SERVICE
    ON RAW_TEXT -- 검색의 대상이 되는 컬럼(벡터 인덱싱) - 입력값의 벡터값을 보고 품목과, 결과값을 유추합니다.
    ATTRIBUTES ID, ITEM_CATEGORY, FINAL_JSON -- 검색 결과와 필터에 사용할 컬럼
    WAREHOUSE = 'WH_CORTEX' -- 웨어하우스 설정
    TARGET_LAG = '1 minute' -- 이벤트 발생시 1분안에 반영
    AS 
    SELECT 
        ID,
        RAW_TEXT, 
        ITEM_CATEGORY, 
        FINAL_JSON 
    FROM {DB}.{SCHEMA}.COTEX_SEARCH_REFER_TABLE;

In [ ]:
# OCR 참조 테이블 생성 예시

import snowflake.snowpark as snowpark
import json
import uuid

def insert_ocr_rag_sample(
    session: snowpark.Session,
    raw_text_data: dict,
    item_category: str,
    final_json_data: dict
    ):  
    
    new_id = str(uuid.uuid4())
    raw_text_str = json.dumps(raw_text_data, indent=4, ensure_ascii=False)
    final_json_str = json.dumps(final_json_data, indent=4, ensure_ascii=False)
    item_category = (item_category or "").strip()

    insert_query = """
    INSERT INTO {DB_NAME}.{SCHEMA_NAME}.OCR_RAG_SAMPLES 
        (ID, RAW_TEXT, ITEM_CATEGORY, FINAL_JSON)
    SELECT ?, ?, ?, ?
    """

    session.sql(
        insert_query,
        params=[new_id, raw_text_str, item_category, final_json_str]
    ).collect()

    return session.create_dataframe(
        [[new_id, "Success", "Data inserted into OCR_RAG_SAMPLES"]],
        schema=["ID", "STATUS", "MESSAGE"],
    )

def get_sample_payload():
# 입력데이터
    raw_text_data = {
      "Header": {
        "Item Category": "레미콘",
        "Title": "레디믹스트 콘크리트 송장",
        "Standard_Info": {
          "표 준 명": "레디믹스트 콘크리트",
          "콘크리트종류": "보통·포장·고강도"
        },
        "Site_Info": {
          "현장": "010-0000-0000",
          "메모": "0번게이트(000동)",
          "감리": "(Signature)",
          "앞차": "0000"
    # (생략)
    }}}

# 품목   
    item_category = "레미콘"

#  출력 결과 예시   
    final_json_data = {
      "car_no": "0000", "car_no_verification_required": False,
      "ship_vme": 6.00, "ship_vme_verification_required": False,
      "vehicle_departure_time": "00 시 00 분", "vehicle_departure_time_verification_required": True,
      "vehicle_arrival_time": "00 시 00 분", "vehicle_arrival_time_verification_required": True,
    # (생략)
    }


    
def main(session: snowpark.Session):
    raw_text_data, item_category, final_json_data = get_sample_payload()
    return insert_ocr_rag_sample(
        session=session,
        raw_text_data=raw_text_data,
        item_category=item_category,
        final_json_data=final_json_data,
    ) 


### 3. OCR 결과테이블 생성
- `ocr_batch.ipynb`의 ocr 결과가 여기에 들어온다.

In [ ]:
-- OCR 결과 테이블 생성
create or replace TABLE {DB}.{SCHEMA}.OCR_RESULT_TABLE (
	ID NUMBER(38,0) autoincrement start 1 increment 1 noorder COMMENT 'ID',
	STAGE_LOCATION VARCHAR(16777216) COMMENT '이미지 저장 URL',
	CD_SITE VARCHAR(16777216) COMMENT '현장코드',
	IMAGE_NAME VARCHAR(16777216) COMMENT '이미지 파일명',
	ITEM_CATEGORY VARCHAR(16777216) COMMENT '품목 카테고리',
	RAW_TEXT VARCHAR(16777216) COMMENT '1차 OCR 결과',
	FINAL_JSON VARCHAR(16777216) COMMENT '2차 OCR 결과',
	IS_SUCCESS BOOLEAN COMMENT '성공여부',
	ERROR_MESSAGE VARCHAR(16777216) COMMENT '실패 시 에러 내용',
	PROCESSED_AT TIMESTAMP_NTZ(9) DEFAULT CAST(CURRENT_TIMESTAMP() AS TIMESTAMP_NTZ(9))
);

### 4. 결과 조회 PROCEDURE 생성
- `OCR_RESULT_TABLE`을 조회하여 OCR 결과를 필요할 때 가져가는 방식으로 구현

In [ ]:
-- 결과를 조회하는 프로시저 생성

CREATE OR REPLACE PROCEDURE {DB}.{SCHEMA}.GET_OCR_RESULT("P_CD_SITE" VARCHAR, "P_IMAGE_NAME" VARCHAR)
RETURNS VARIANT
LANGUAGE SQL
EXECUTE AS OWNER
AS '
BEGIN
    LET RES VARIANT := (
        SELECT COALESCE(
            (
                SELECT OBJECT_CONSTRUCT(
                    ''SUCCESS'', IS_SUCCESS,
                    ''MESSAGE'', IFF(IS_SUCCESS = TRUE, '''', COALESCE(ERROR_MESSAGE, ''알 수 없는 원인으로 OCR 처리에 실패했습니다.'')),
                    ''DATA'', IFF(IS_SUCCESS = TRUE, TRY_PARSE_JSON(FINAL_JSON), OBJECT_CONSTRUCT()),
                    ''PROCESSED_AT'', PROCESSED_AT
                )::VARIANT
                FROM (
                    SELECT *
                    FROM {DB}.{SCHEMA}.OCR_RESULT_TABLE
                    WHERE CD_SITE = :P_CD_SITE
                      AND IMAGE_NAME = :P_IMAGE_NAME
                    ORDER BY PROCESSED_AT DESC
                    LIMIT 1
                )
            ),
            OBJECT_CONSTRUCT(
                ''SUCCESS'', FALSE,
                ''MESSAGE'', ''요청하신 현장코드와 이미지명에 해당하는 데이터가 존재하지 않습니다.'',
                ''DATA'', OBJECT_CONSTRUCT(),
                ''PROCESSED_AT'', NULL
            )::VARIANT
        )
    );
    
    RETURN RES;
END;
';

### 결과 CALL 예시

In [ ]:
CALL {DB}.{SCHEMA}.GET_OCR_RESULT('SITE_A', 'invoice_20260220_01.jpg');

### 추후에 할 일
- `ocr_batch.ipynb`를 프로시저로 만들어서, Task를 통해 일정 시간만큼 호출하면 Stream이 마지막 호출을 기준으로 적재된 image를 병렬로 OCR함